# 10 – Export the Deployment Model for the RoadLens App

**Component 4 – Multi-Sensor ML Road Roughness Assessment and Cross-Vehicle-Driver Generalization**
Sandanayaka S D I K – IT23222472

Notebook 07 saved the smartphone deployment model (configuration **C7**: one dashboard sensor, four
orientation-invariant channels, 40 cross-vehicle stable features, XGBoost; LOVO macro-F1 ≈ 0.83) as
`PVS_Model_Results_v2/c4_deployment_model.joblib`.

A joblib file stores Python objects (a scikit-learn pipeline). It can only be loaded reliably with the
**same scikit-learn and xgboost versions** that created it, while the RoadLens backend uses its own pinned
versions. This notebook converts the trained model into a **portable format** that any recent xgboost
version can read:

| File | Content |
|---|---|
| `c4_model_portable.json` | features, imputer medians, scaler mean/std, speed-normalisation slopes, pipeline configuration, classes, metadata |
| `c4_model_portable.xgb.json` | the trained XGBoost trees (xgboost's own JSON format) |

The model itself is **not retrained or changed**. Section 5 checks that the portable model gives the
same class probabilities as the original joblib model.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Locate the deployment model

In [2]:
import os, json, hashlib
import numpy as np
import joblib, sklearn, xgboost

RESULTS_DIR = '/content/drive/MyDrive/PVS_Model_Results_v2'
BUNDLE_PATH = os.path.join(RESULTS_DIR, 'c4_deployment_model.joblib')
EXPORT_DIR = os.path.join(RESULTS_DIR, 'portable')

print('Bundle found :', os.path.exists(BUNDLE_PATH), BUNDLE_PATH)
print('scikit-learn :', sklearn.__version__)
print('xgboost      :', xgboost.__version__)

Bundle found : True /content/drive/MyDrive/PVS_Model_Results_v2/c4_deployment_model.joblib
scikit-learn : 1.6.1
xgboost      : 3.4.1


## 3. Load and inspect the bundle

In [3]:
bundle = joblib.load(BUNDLE_PATH)
pipe = bundle['model']
steps = dict(pipe.named_steps)

print('Keys          :', sorted(bundle))
print('Model         :', bundle.get('model_name'))
print('Pipeline      :', list(steps))
print('Sensor        :', bundle.get('sensor'))
print('Channels      :', bundle.get('channels'))
print('Features      :', len(bundle['features']))
print('Speed slopes  :', len(bundle.get('speed_slopes') or {}))
print('Classes       :', bundle.get('classes'))
print('LOVO macro-F1 :', bundle.get('lovo_macro_f1'))
print('Notes         :', bundle.get('notes'))
assert set(steps) == {'simpleimputer', 'standardscaler', 'xgbclassifier'}, 'Unexpected pipeline steps'
for f in bundle['features'][:10]:
    print('  ', f)

Keys          : ['channels', 'classes', 'config', 'features', 'lovo_macro_f1', 'model', 'model_name', 'notes', 'sensor', 'speed_ref_log', 'speed_slopes']
Model         : XGB
Pipeline      : ['simpleimputer', 'standardscaler', 'xgbclassifier']
Sensor        : left_dashboard
Channels      : ['acc_vert', 'acc_horiz', 'acc_mag', 'gyro_mag']
Features      : 40
Speed slopes  : 27
Classes       : ['good', 'regular', 'bad']
LOVO macro-F1 : 0.8278993248670137
Notes         : Per-trip normalisation statistics and the stationary noise profile are computed on each incoming trip.
   t__left_dashboard__acc_mag__mean
   t__left_dashboard__gyro_mag__peak
   t__left_dashboard__gyro_mag__rms
   t__left_dashboard__gyro_mag__iqr
   f__left_dashboard__gyro_mag__bprel_0.5-3
   f__left_dashboard__gyro_mag__entropy
   f__left_dashboard__gyro_mag__centroid
   f__left_dashboard__acc_vert__entropy
   f__left_dashboard__acc_mag__bprel_16-30
   f__left_dashboard__gyro_mag__bprel_16-30


## 4. Export to the portable format

The exported JSON contains only plain numbers, strings and lists. The pipeline configuration (`config`)
is stored so that the backend applies **exactly the same** windowing, filtering and thresholds as training.

In [8]:
def plain(o):
    """Convert numpy / tuple values into JSON-safe Python values."""
    if isinstance(o, dict):
        return {str(k): plain(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [plain(v) for v in o]
    if isinstance(o, np.generic):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    return o

imp, sc, clf = steps['simpleimputer'], steps['standardscaler'], steps['xgbclassifier']

# How StandardScaler rounds float32 input depends on the scikit-learn version: older versions subtract and
# divide with the float64 mean/scale step by step (X -= mean_; X /= scale_), newer versions first cast the
# parameters to float32. The difference is one float32 rounding step, but it can move a value across a tree
# split, so the exact arithmetic of THIS model is detected and stored.
Xs = np.random.default_rng(1).standard_normal((500, len(bundle['features']))).astype(np.float32)
ref = sc.transform(imp.transform(Xs))
per_step = Xs.copy(); per_step -= sc.mean_; per_step /= sc.scale_
f32_params = (Xs - sc.mean_.astype(np.float32)) / sc.scale_.astype(np.float32)
scale_arithmetic = ('float64_per_step' if np.array_equal(per_step, ref)
                    else 'float32_params' if np.array_equal(f32_params, ref) else None)
assert scale_arithmetic, 'Scaler arithmetic could not be reproduced'
print('Scaler arithmetic :', scale_arithmetic)

os.makedirs(EXPORT_DIR, exist_ok=True)
booster_file = 'c4_model_portable.xgb.json'
clf.get_booster().save_model(os.path.join(EXPORT_DIR, booster_file))

meta = plain({k: v for k, v in bundle.items() if k != 'model'})
lovo = bundle.get('lovo_macro_f1') or 0
meta.update(
    model_version=f"{bundle.get('model_name', 'XGB')}-C7-lovo{lovo:.3f}",
    demo=False,
    booster_file=booster_file,
    impute_median=plain(imp.statistics_),
    scale_mean=plain(sc.mean_),
    scale_std=plain(sc.scale_),
    scale_arithmetic=scale_arithmetic,
    exported_with={'scikit-learn': sklearn.__version__, 'xgboost': xgboost.__version__},
)
with open(os.path.join(EXPORT_DIR, 'c4_model_portable.json'), 'w') as f:
    json.dump(meta, f, indent=1, allow_nan=False)

for name in ('c4_model_portable.json', booster_file):
    p = os.path.join(EXPORT_DIR, name)
    print(f'{name:30s} {os.path.getsize(p)/1024:8.1f} KB  sha256 {hashlib.sha256(open(p, "rb").read()).hexdigest()[:16]}…')
print('Model version     :', meta['model_version'])

Scaler arithmetic : float64_per_step
c4_model_portable.json              7.4 KB  sha256 553fdcf644ece7de…
c4_model_portable.xgb.json       2415.3 KB  sha256 ae49693aaf0edc9e…
Model version     : XGB-C7-lovo0.828


## 5. Check that the portable model reproduces the original model

The portable model applies the same three steps as the scikit-learn pipeline — median imputation,
standard scaling and the XGBoost trees — with the same float32 rounding as the trained pipeline. Both models are compared on
2,000 random feature vectors (with 2 % missing values).

In [9]:
def portable_predict(X, meta, booster_path):
    """Imputation + scaling + XGBoost trees, with the same float32 rounding as the trained pipeline."""
    f32 = np.float32
    booster = xgboost.Booster()
    booster.load_model(booster_path)
    fill = np.asarray(meta['impute_median'], float)
    mean = np.asarray(meta['scale_mean'], float)
    std = np.asarray(meta['scale_std'], float)
    std = np.where(std == 0, 1, std)
    X = np.asarray(X, f32)
    X = np.where(np.isfinite(X), X, fill.astype(f32))
    if meta['scale_arithmetic'] == 'float64_per_step':
        X = (X - mean).astype(f32)
        X = (X / std).astype(f32)
    else:
        X = (X - mean.astype(f32)) / std.astype(f32)
    return booster.predict(xgboost.DMatrix(X))

rng = np.random.default_rng(0)
X = rng.standard_normal((2000, len(bundle['features']))).astype(np.float32) * 2
X[rng.random(X.shape) < 0.02] = np.nan
p_joblib = pipe.predict_proba(X)
p_portable = portable_predict(X, meta, os.path.join(EXPORT_DIR, booster_file))

max_diff = float(np.abs(p_joblib - p_portable).max())
agree = float((p_joblib.argmax(1) == p_portable.argmax(1)).mean())
print(f'Max probability difference : {max_diff:.2e}')
print(f'Same predicted class       : {agree:.2%}')
assert max_diff < 1e-5, 'Portable export does not reproduce the joblib model'
print('OK - the portable model is identical to the trained model.')

Max probability difference : 0.00e+00
Same predicted class       : 100.00%
OK - the portable model is identical to the trained model.


## 6. Download the two files

The files are also kept in `PVS_Model_Results_v2/portable/` on Drive.

In [10]:
from google.colab import files
files.download(os.path.join(EXPORT_DIR, 'c4_model_portable.json'))
files.download(os.path.join(EXPORT_DIR, booster_file))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 7. Use the model in RoadLens

1. Copy both files into the repository folder `ml/c4/models/production/`. Model weights are not committed
   to Git: this folder is ignored.
2. Rebuild and start the platform: `docker compose up --build`.
3. Open `http://localhost:3000/c4`. The badge changes from *Demo model (synthetic)* to **Production model**,
   and every result record carries `source_kind: "inference"` and this model version with the SHA-256 of
   the model files.